# RCT Results Analysis

## Setup

In [ ]:
# whether to use the private version of the data 
# or the published version (with researcher/evaluator names pseudonymized):
use_private_data = False

In [ ]:
# whether to retrieve and save data on Google Sheets / Google Drive 
# or from and to the local drive
use_Google_Drive = False

if use_Google_Drive:
    from google.colab import auth
    import gspread
    from google.auth import default
else:
    from pathlib import Path

In [ ]:
import pandas as pd
import numpy as np
import traceback
from collections import Counter
import re

In [ ]:
if use_Google_Drive:
    # 1. Authenticate the user
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    
    # 2. Open the spreadsheet by ID
    spreadsheet_id = "SpreadSheetId"
    try:
        # Open the spreadsheet
        sh = gc.open_by_key(spreadsheet_id)
        # Get the first worksheet
        worksheet = sh.get_worksheet(0)
    
        # 3. Get all values and convert to DataFrame
        data = worksheet.get_all_values()
        if not data:
            print('The sheet is empty.')
        else:
            df_raw = pd.DataFrame(data[1:], columns=data[0])
            print(f'Successfully retrieved {len(df_raw)} rows from the Google Sheet.')
            display(df_raw.head())
    except Exception as e:
        print('An error occurred during execution:')
        traceback.print_exc()

else: 
    DATA_DIR = Path("../data/")
    INPUT_FILENAME = "RCT_responses_cleaned.csv"
    INPUT_CSV = DATA_DIR / INPUT_FILENAME
    OUT_DIR = Path("./analysis_outputs/")
    OUT_DIR.mkdir(exist_ok=True)
    
    df_raw = pd.read_csv(INPUT_CSV)

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)
pd.set_option("display.expand_frame_repr", False)


print(f"Rows: {df_raw.shape[0]}")
print(f"Columns: {df_raw.shape[1]}")

## Clean Key Fields

In [ ]:
COL_CONDITION = "Condition"
COL_DOMAIN = "Domain"

# When using the private spreadsheet as source, use the non-anonymized evaluator name column:
if use_private_data:
    COL_RESEARCHER = "Human Researcher [your name]"
else: 
    # When using the published evaluation data as source, use the equivalent anonymized evaluator name column:
    COL_RESEARCHER = "evaluator"

COL_MATCH = "Results: Match"
COL_FAILURE = "Reproduction failure mode classification (in case reproduction of the given target failed)"

# Define a function to clean text values by stripping leading/trailing whitespace and handling NaN values
def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()

df = df_raw.copy()

case_key_cols = [COL_CONDITION, COL_RESEARCHER, "Paper Title"]

# Check for duplicate case keys (Condition + Human Researcher + Paper Title)
duplicate_case_keys = df.duplicated(subset=case_key_cols, keep=False)
if duplicate_case_keys.any():
    raise ValueError("Case ID key is not unique. Check duplicate Condition + "+COL_RESEARCHER+" + Paper Title rows.")

# Sort the DataFrame by the case key columns to ensure consistent ordering
df = df.sort_values(case_key_cols).reset_index(drop=False).rename(columns={"index": "original_row_id"})

# Assign a unique case ID to each row based on the combination of Condition, Human Researcher, and Paper Title
df["case_id"] = range(1, len(df) + 1)

case_lookup = df[["case_id", COL_CONDITION, COL_RESEARCHER, "Paper Title", COL_DOMAIN]].copy()

## Data Overview

In [ ]:
overview = pd.DataFrame({
    "rows": [len(df)],
    "columns": [df.shape[1]],
    "unique_papers": [df["Paper Title"].nunique()],
    "unique_researchers": [df[COL_RESEARCHER].nunique()],
    "domains": [df[COL_DOMAIN].nunique()],
    "conditions": [df[COL_CONDITION].nunique()],
})
overview

condition_domain_balance = pd.crosstab(df[COL_DOMAIN], df[COL_CONDITION], margins=True)
condition_domain_balance


## Create Step-Level Summary

In [ ]:
STEP_DEFINITIONS = [
    {
        "step": "1.1",
        "step_name": "Start the instance and Docker image",
        "who_default": "Human",
        "outcome_col": "Step 1.1: Start the instance and Docker image (Human) - Outcome",
    },
    {
        "step": "1.2",
        "step_name": "Start Agent with logging and prompt replication target task",
        "who_default": "Human",
        "outcome_col": "Step 1.2: Start Agent with logging and prompt replication target task (using the generic default prompt) (Human) - Outcome",
    },
    {
        "step": "1.3",
        "step_name": "Obtain the paper's code",
        "who_col": "Step 1.3: Obtain the paper’s code (e.g. clone repo) - Who did it",
        "outcome_col": "Step 1.3: Obtain the paper’s code (e.g. clone repo) - Outcome",
    },
    {
        "step": "1.4",
        "step_name": "Read README",
        "who_col": "Step 1.4: Read README - Who did it",
        "outcome_col": "Step 1.4: Read README - Outcome",
    },
    {
        "step": "1.5",
        "step_name": "Create environment",
        "who_col": "Step 1.5: Create environment (e.g. using conda/venv) -- Who did it",
        "outcome_col": "Step 1.5: Create environment (e.g. using conda/venv) - Outcome",
    },
    {
        "step": "1.6",
        "step_name": "Install dependencies",
        "who_col": "Step 1.6: Install dependencies - Who did it",
        "outcome_col": "Step 1.6: Install dependencies - Outcome",
    },
    {
        "step": "1.7",
        "step_name": "Download/prepare data",
        "who_col": "Step 1.7: Download/prepare data - Who did it",
        "outcome_col": "Step 1.7: Download/prepare data - Outcome",
    },
    {
        "step": "1.8",
        "step_name": "Verify setup",
        "who_col": "Step 1.8: Verify setup (import test, etc.) - Who did it",
        "outcome_col": "Step 1.8: Verify setup (import test, etc.) - Outcome",
    },
    {
        "step": "2.1",
        "step_name": "Identify entry point / main script",
        "who_col": "Step 2.1: Identify entry point / main script - Who did it",
        "outcome_col": "Step 2.1: Identify entry point / main script - Outcome",
        "combined_col": "Phase 2 *Select one from \"Who\" and one from \"Outcome\" [2.1 Identify entry point / main script]",
    },
    {
        "step": "2.2",
        "step_name": "Understand required run parameters",
        "who_col": "Step 2.2: Understand required run parameters - Who did it",
        "outcome_col": "Step 2.2: Understand required run parameters - Outcome",
        "combined_col": "Phase 2 *Select one from \"Who\" and one from \"Outcome\" [2.2 Understand required run parameters]",
    },
    {
        "step": "2.3",
        "step_name": "Run code",
        "who_col": "Step 2.3: Run code - Who did it",
        "outcome_col": "Step 2.3: Run code - Outcome",
        "combined_col": "Phase 2 *Select one from \"Who\" and one from \"Outcome\" [2.3 Run Code]",
    },
    {
        "step": "2.4",
        "step_name": "Monitor / debug runtime errors",
        "who_col": "Step 2.4: Monitor / debug runtime errors - Who did it",
        "outcome_col": "Step 2.4: Monitor / debug runtime errors - Outcome",
        "combined_col": "Phase 2 *Select one from \"Who\" and one from \"Outcome\" [2.4 Monitor / debug runtime errors]",
    },
    {
        "step": "2.5",
        "step_name": "Locate output files",
        "who_col": "Step 2.5: Locate output files - Who did it",
        "outcome_col": "Step 2.5: Locate output files - Outcome",
        "combined_col": "Phase 2 *Select one from \"Who\" and one from \"Outcome\" [2.5 Locate output files]",
    },
    {
        "step": "3.1",
        "step_name": "Parse/extract our results",
        "who_col": "Step 3.1: Parse/extract our results - Who did it",
        "outcome_col": "Step 3.1: Parse/extract our results - Outcome",
        "combined_col": "Phase 3 *Select one from \"Who\" and one from \"Outcome\" [3.1 Parse / extract our results]",
    },
    {
        "step": "3.2",
        "step_name": "Compare to paper values",
        "who_col": "Step 3.2: Compare to paper values - Who did it",
        "outcome_col": "Step 3.2: Compare to paper values - Outcome",
        "combined_col": "Phase 3 *Select one from \"Who\" and one from \"Outcome\" [3.2 Compare to paper values]",
    },
    {
        "step": "3.3",
        "step_name": "Investigate discrepancies",
        "who_col": "Step 3.3 - Investigate discrepancies (if any) - Who did it",
        "outcome_col": "Step 3.3 - Investigate discrepancies (if any) - Outcome",
        "combined_col": "Phase 3 *Select one from \"Who\" and one from \"Outcome\" [3.3 Investigate discrepancies(if any)]",
    },
]

# Clean and normalize outcome values to a consistent set of categories (Success, Partial success, Fail, N/A, Missing)
def normalize_outcome(value):
    text = clean_text(value)
    if not text:
        return "Missing"
    lower = text.lower()
    if "partial" in lower:
        return "Partial success"
    if "success" in lower or "succcess" in lower:
        return "Success"
    if "fail" in lower:
        return "Fail"
    if "n/a" in lower or "not applicable" in lower:
        return "N/A"
    return text

def normalize_who(value):
    text = clean_text(value)
    return text if text else "Missing"

def parse_combined_who_outcome(value):
    text = clean_text(value)
    parsed = {"who": "Missing", "outcome": "Missing"}
    if not text:
        return parsed
    for part in text.split(","):
        key, sep, raw_value = part.partition(":")
        if not sep:
            continue
        key = key.strip().lower()
        raw_value = raw_value.strip()
        if key == "who":
            parsed["who"] = normalize_who(raw_value)
        elif key == "outcome":
            parsed["outcome"] = normalize_outcome(raw_value)
    return parsed

# Iterate through each row and step definition to create a long-format DataFrame
# where each row corresponds to a specific step for a specific case, with the associated who and outcome information.
step_rows = []
for row_id, row in df.iterrows():
    for step_def in STEP_DEFINITIONS:
        outcome_col = step_def["outcome_col"]
        if outcome_col not in df.columns and step_def.get("combined_col") not in df.columns:
            continue
        who_col = step_def.get("who_col")
        who_recorded = False
        if who_col is None:
            who = step_def.get("who_default", "Missing")
        else:
            who_raw = row.get(who_col, "")
            who = normalize_who(who_raw)
            who_recorded = bool(clean_text(who_raw))
        outcome = normalize_outcome(row.get(outcome_col, ""))

        if outcome == "Missing" and step_def.get("combined_col") in df.columns:
            combined = parse_combined_who_outcome(row.get(step_def["combined_col"], ""))
            if combined["who"] != "Missing":
                who = combined["who"]
                who_recorded = True
            outcome = combined["outcome"]

        if outcome == "Missing" and not who_recorded:
            continue
        step_rows.append({
            "case_id": row["case_id"],
            "paper_title": row["Paper Title"],
            "human_researcher": row[COL_RESEARCHER],
            "domain": row[COL_DOMAIN],
            "step": step_def["step"],
            "step_name": step_def["step_name"],
            "who_did_it": who,
            "outcome": outcome,
        })

step_long = pd.DataFrame(step_rows)

## Data Clean Up

In [ ]:
# Manual correction confirmed after reviewing the older questionnaire response:
# Human Researcher A's AI-assisted decentralization case recorded who did steps 3.1-3.3,
# but the older form did not capture the corresponding outcome cells. These were
# confirmed as Success and are filled here so the step-level table is complete.
older_form_success_fix = (
    step_long["case_id"].eq(1)
    & step_long["human_researcher"].eq("A")
    & step_long["paper_title"].eq("decentralization can increase cooperation among public officials")
    & step_long["step"].isin(["3.1", "3.2", "3.3"])
    & step_long["outcome"].eq("Missing")
)
step_long.loc[older_form_success_fix, "outcome"] = "Success"

# Reclassify partial-success steps that were actually resolved through
# human-agent collaboration. The original form marked these as Agent / Partial
# success, but blocker notes show human suggestion, intervention, or approval
# was needed for the step to succeed.
human_resolved_partial_success_keys = pd.DataFrame([
    {"case_id": 7, "step": "1.6", "reclassification_reason": "Python dependency conflict resolved after human suggestion/intervention to downgrade to Python 3.6/3.7."},
    {"case_id": 7, "step": "2.3", "reclassification_reason": "Run succeeded after human intervention to use an older Python environment compatible with the archived suite."},
    {"case_id": 25, "step": "2.4", "reclassification_reason": "Runtime error was resolved after human approval to switch to the older R version specified in the README."},
])
reclass_key = list(zip(human_resolved_partial_success_keys["case_id"], human_resolved_partial_success_keys["step"]))
reclass_mask = (
    step_long[["case_id", "step"]].apply(tuple, axis=1).isin(reclass_key)
    & step_long["who_did_it"].eq("Agent")
    & step_long["outcome"].eq("Partial success")
)

# Table of steps that are being reclassified with the reason for reclassification
partial_to_both_success_before_after = step_long.loc[
    reclass_mask,
    ["case_id", "paper_title", "human_researcher", "domain", "step", "step_name", "who_did_it", "outcome"],
].merge(human_resolved_partial_success_keys, on=["case_id", "step"], how="left")
partial_to_both_success_before_after = partial_to_both_success_before_after.rename(columns={
    "who_did_it": "before_who_did_it",
    "outcome": "before_outcome",
})
partial_to_both_success_before_after["after_who_did_it"] = "Both"
partial_to_both_success_before_after["after_outcome"] = "Success"

In [ ]:
# Apply the reclassification to the main step_long DataFrame
step_long.loc[reclass_mask, "who_did_it"] = "Both"
step_long.loc[reclass_mask, "outcome"] = "Success"
step_long[["step", "step_name", "who_did_it", "outcome"]].drop_duplicates().sort_values(["step", "who_did_it", "outcome"])


## Count of Outcome by Case

In [ ]:
# Create a summary table that counts the number of cases for each combination of step, who_did_it, and outcome
step_summary = (
    step_long.groupby(["step", "step_name", "who_did_it", "outcome"])
    .size()
    .reset_index(name="n")
    .sort_values(["step", "who_did_it", "outcome"])
)

## Result Table

In [ ]:
# Create a pivot table to show the count of cases for each step, broken down by who_did_it and outcome
step_outcome_pivot = pd.pivot_table(
    step_long,
    index=["step", "step_name"],
    columns=["who_did_it", "outcome"],
    values="case_id",
    aggfunc="count",
    fill_value=0,
)
standard_who = ["Agent", "Both", "Human"]
standard_outcomes = ["Success", "Partial success", "Fail"]
standard_columns = pd.MultiIndex.from_product([standard_who, standard_outcomes], names=["who_did_it", "outcome"])
extra_columns = step_outcome_pivot.columns.difference(standard_columns)
step_outcome_pivot = step_outcome_pivot.reindex(columns=standard_columns.append(extra_columns), fill_value=0)
step_outcome_pivot

## Blocker Summary in Questionnaire Responses

In [ ]:
# Blocker summary in questionnaire responses (distinct from Docent "Blocker" logs)
blocker_rows = []
for phase in ["Phase 1", "Phase 2", "Phase 3"]:
    for blocker_idx in [1, 2, 3]:
        blocker_col = f"{phase} Blocker {blocker_idx}: What was the Blocker"
        who_col = f"{phase} Blocker {blocker_idx}: Who got stuck"
        intervention_col = f"{phase} Blocker {blocker_idx}: Intervention needed?"
        if intervention_col not in df.columns:
            intervention_col = f"{phase} Blocker {blocker_idx}:Intervention needed?"
        if blocker_col not in df.columns:
            continue

        has_blocker = df[blocker_col].map(clean_text).ne("")
        for outcome, subset in df.loc[has_blocker].groupby(COL_MATCH, dropna=False):
            blocker_rows.append({
                "phase": phase,
                "blocker_slot": blocker_idx,
                "outcome": clean_text(outcome),
                "n_blockers_recorded": len(subset),
                "agent_stuck_n": int(subset[who_col].map(clean_text).str.contains("Agent", case=False).sum()) if who_col in df.columns else np.nan,
                "intervention_needed_yes_n": int(subset[intervention_col].map(clean_text).str.contains("Yes", case=False).sum()) if intervention_col in df.columns else np.nan,
            })

blocker_summary = pd.DataFrame(blocker_rows).sort_values(["phase", "blocker_slot", "outcome"])


In [ ]:
STEP_NOTE_COLUMNS = {
    "1.1": "Step 1.1: Start the instance and Docker image (Human) - Notes",
    "1.3": "Step 1.3: Obtain the paper’s code (e.g. clone repo) - Notes",
    "1.4": "Step 1.4: Read README - Notes",
    "1.5": "Step 1.5: Create environment (e.g. using conda/venv) -- Notes",
    "1.6": "Step 1.6: Install dependencies - Notes",
    "1.7": "Step 1.7: Download/prepare data - Notes",
    "1.8": "Step 1.8: Verify setup (import test, etc.) - Notes",
    "2.1": "Step 2.1: Identify entry point / main script - Notes",
    "2.3": "Step 2.3: Run code - Notes",
    "2.4": "Step 2.4: Monitor / debug runtime errors - Notes",
    "2.5": "Step 2.5: Locate output files - Notes",
    "3.1": "Step 3.1: Parse/extract our results - Notes",
    "3.2": "Step 3.2: Compare to paper values - Notes",
    "3.3": "Step 3.3 - Investigate discrepancies (if any) - Notes",
}

# For easier lookup later, create a mapping of step to the corresponding notes column in the original DataFrame
BLOCKER_SPECS = []
for phase in ["Phase 1", "Phase 2", "Phase 3"]:
    for slot in [1, 2, 3]:
        blocker_col = f"{phase} Blocker {slot}: What was the Blocker"
        who_col = f"{phase} Blocker {slot}: Who got stuck"
        resolution_col = f"{phase} Blocker {slot}: What was the resolution"
        intervention_col = f"{phase} Blocker {slot}: Intervention needed?"
        if intervention_col not in df.columns:
            intervention_col = f"{phase} Blocker {slot}:Intervention needed?"
        if blocker_col in df.columns:
            BLOCKER_SPECS.append((phase, slot, blocker_col, who_col, resolution_col, intervention_col))

# Define a function to retrieve the original cell value for a given row and column, handling missing values
def original_cell(row_id, col):
    if not col or col not in df.columns or pd.isna(df.loc[row_id, col]):
        return ""
    return str(df.loc[row_id, col]).strip()

# Define a function to collect blocker records for a given row ID, iterating through the defined BLOCKER_SPECS and retrieving the relevant information for each recorded blocker.
def collect_blocker_records(row_id):
    records = []
    for phase, slot, blocker_col, who_col, resolution_col, intervention_col in BLOCKER_SPECS:
        blocker = original_cell(row_id, blocker_col)
        if not blocker:
            continue
        records.append({
            "phase": phase,
            "slot": slot,
            "blocker": blocker,
            "who_got_stuck": original_cell(row_id, who_col),
            "resolution": original_cell(row_id, resolution_col),
            "intervention_needed": original_cell(row_id, intervention_col),
        })
    return records

# Define a function to format blocker records into a readable string format for display or reporting purposes.
def format_blocker_records(blockers):
    def one_line(value):
        return " ".join(str(value).split())

    chunks = []
    for blocker in blockers:
        chunks.append(
            f"{one_line(blocker['phase'])} Blocker {blocker['slot']} | "
            f"Who got stuck: {one_line(blocker['who_got_stuck'])} | "
            f"Intervention needed: {one_line(blocker['intervention_needed'])} | "
            f"Blocker: {one_line(blocker['blocker'])} | "
            f"Resolution: {one_line(blocker['resolution'])}"
        )
    return " || ".join(chunks)

case_id_to_df_index = pd.Series(df.index, index=df["case_id"]).to_dict()

# Define a function to retrieve the original cell value for a given case ID, step, and column by looking up the corresponding row in the original DataFrame and returning the cleaned text value from the specified column.
def step_result_for_case(case_id, step, column):
    matches = step_long.loc[
        step_long["case_id"].eq(case_id) & step_long["step"].eq(step),
        column,
    ]
    if matches.empty:
        return ""
    return matches.iloc[0]


## Review Table of Agent Partial Success or Both-Success

In [ ]:
target_step_records = step_long[
    step_long["outcome"].eq("Partial success")
    | (step_long["who_did_it"].eq("Both") & step_long["outcome"].eq("Success"))
].copy()

partial_or_both_success_rows = []
for _, step_record in target_step_records.iterrows():
    row_id = int(case_id_to_df_index[step_record["case_id"]])
    step = str(step_record["step"])
    blockers = collect_blocker_records(row_id)
    partial_or_both_success_rows.append({
        "case_id": step_record["case_id"],
        "paper_title": step_record["paper_title"],
        "human_researcher": step_record["human_researcher"],
        "domain": step_record["domain"],
        "step": step,
        "step_name": step_record["step_name"],
        "who_did_it": step_record["who_did_it"],
        "outcome": step_record["outcome"],
        "results_question": original_cell(row_id, "Results: Question"),
        "results_paper_value": original_cell(row_id, "Results: Paper Value"),
        "results_our_value": original_cell(row_id, "Results: Our Value"),
        "results_match": original_cell(row_id, "Results: Match"),
        "step_3_1_who_did_it": step_result_for_case(step_record["case_id"], "3.1", "who_did_it"),
        "step_3_1_outcome": step_result_for_case(step_record["case_id"], "3.1", "outcome"),
        "step_3_2_who_did_it": step_result_for_case(step_record["case_id"], "3.2", "who_did_it"),
        "step_3_2_outcome": step_result_for_case(step_record["case_id"], "3.2", "outcome"),
        "step_3_3_who_did_it": step_result_for_case(step_record["case_id"], "3.3", "who_did_it"),
        "step_3_3_outcome": step_result_for_case(step_record["case_id"], "3.3", "outcome"),
        "reproduction_failure_mode": original_cell(row_id, COL_FAILURE),
        "step_note": original_cell(row_id, STEP_NOTE_COLUMNS.get(step, "")),
        "n_blockers": len(blockers),
        "blocker_details": format_blocker_records(blockers),
    })

partial_or_both_success_notes = pd.DataFrame(partial_or_both_success_rows).sort_values(["case_id", "step"])
print(partial_or_both_success_notes.shape)

In [ ]:
def join_unique(values):
    cleaned = [clean_text(value) for value in values if clean_text(value)]
    return " || ".join(dict.fromkeys(cleaned))


def summarize_steps(group):
    step_labels = group.sort_values("step").apply(lambda row: f"{row['step']} {row['step_name']}", axis=1)
    return "; ".join(step_labels)

# Summarize the steps that were marked as Partial success or Both-success for each case

summary_rows = []
# Iterate through each case to create another table with summary of
for case_id, group in partial_or_both_success_notes.groupby("case_id", sort=True):
    row = group.iloc[0]
    classification = "Partial success" if group["outcome"].eq("Partial success").any() else "Both success"
    summary_rows.append({
        "classification": classification,
        "case_id": int(case_id),
        "paper_title": row["paper_title"],
        "docent_log_link": original_cell(int(case_id_to_df_index[case_id]), "Link to Docent log of this session"),
        "steps": summarize_steps(group),
        "analysis_issue_detail": join_unique([
            join_unique(group["step_note"]),
            join_unique(group["reproduction_failure_mode"]),
        ]),
        "analysis_solution_attempt_detail": join_unique(group["blocker_details"]),
        "final_result": row["results_match"]
        })

partial_or_both_case_summary = pd.DataFrame(summary_rows)

In [ ]:
# merge partial_or_both_case_summary and partial_or_both_success_notes
partial_or_both_case_summary = partial_or_both_case_summary.merge(partial_or_both_success_notes, on=["case_id","paper_title"], how="left")

## Export Tables

In [ ]:
def save_file_in_gdrive(df,file_name):

  from googleapiclient.discovery import build
  from gspread_dataframe import set_with_dataframe

  # 1. Setup Drive API to handle folder placement
  drive_service = build('drive', 'v3', credentials=creds)
  folder_id = 'FolderId'

  # 2. Create the Spreadsheet file metadata
  file_metadata = {
      'name': file_name,
      'parents': [folder_id],
      'mimeType': 'application/vnd.google-apps.spreadsheet'
  }

  try:
      # Create the sheet in the specific folder
      res = drive_service.files().create(body=file_metadata, fields='id').execute()
      new_sheet_id = res.get('id')

      # 3. Open the newly created sheet and write df_cleaned/
      new_sh = gc.open_by_key(new_sheet_id)
      worksheet = new_sh.get_worksheet(0)

      # Use set_with_dataframe for efficient uploading
      set_with_dataframe(worksheet, df)

      print(f'Successfully saved cleaned data to Google Sheet.')
      print(f'Link: https://docs.google.com/spreadsheets/d/{new_sheet_id}')
      print()

  except Exception as e:
      print(f'An error occurred: {e}')

In [ ]:
def save_file_locally(df,file_name):

    output_path = OUT_DIR / f"{file_name}.csv"
    
    df.to_csv(output_path, index=False)

    try:
        print(f'Successfully saved cleaned data locally.')
        print(f'File name: {output_path}')
        print()

    except Exception as e:
      print(f'An error occurred: {e}')

In [ ]:
def save_file(df, file_name):
    if use_Google_Drive:
        save_file_in_gdrive(df, file_name)
    else:
        save_file_locally(df, file_name)

In [ ]:
save_file(case_lookup,'case_lookup')
save_file(overview,'overview')
save_file(condition_domain_balance,'condition_domain_balance')
save_file(step_long,'step_long')
save_file(partial_to_both_success_before_after,'partial_to_both_success_before_after')
save_file(step_summary,'step_summary')
save_file(step_outcome_pivot,'step_outcome_pivot')
save_file(blocker_summary,'blocker_summary')
save_file(partial_or_both_success_notes,'partial_or_both_success_notes')
save_file(partial_or_both_case_summary,'partial_or_both_case_summary')

EoF